# 16. 파생변수 재점검 - 타겟인코딩 & 제외피처 재검토

`파생 변수 생성 (0911 ver)` Notion 문서와 `docs/파생변수_재점검_제안_0911.md` 제안서 내용을 검증하는 노트북입니다.

- 기존 19개 파생변수의 이진 임계값들은 그대로 둡니다 (재설계 시도했으나 효과 없음이 확인됨, #15 참고).
- 대신 (1) 상관계수 기준으로 제외했던 6개 피처 재투입, (2) medical_history x family_medical_history 타겟인코딩
  두 가지를 각각/함께 테스트해서 실제로 CV가 개선되는지 확인합니다.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import os

RANDOM_STATE = 42

# 1. 데이터 로드 및 결측치/중복행 처리 (기존 노트북과 동일)
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

train = train.drop_duplicates(subset=[c for c in train.columns if c != 'ID']).reset_index(drop=True)
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)
for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')
train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

print('train:', train.shape, '/ test:', test.shape)

train: (2994, 18) / test: (3000, 17)


In [2]:
# 2. 파생변수 생성 - 기존 19개는 그대로 (임계값 재설계는 #15에서 효과 없음 확인됨)
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)

    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']

    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)
    return data


# 3. 0911 문서 5번(제외 목록)에서 |r|<0.01로 제외됐던 6개.
# #15/제안서에서 상관계수와 무관하게 LightGBM 기준 CV가 소폭 개선됨을 확인해 재도입 테스트
def add_reinstated_features(df):
    data = df.copy()
    data['is_obese'] = (data['bmi'] >= 30).astype(int)
    data['high_glucose'] = (data['glucose'] >= 126).astype(int)
    data['high_cholesterol'] = (data['cholesterol'] >= 240).astype(int)
    data['metabolic_risk_score'] = data['is_obese'] + data['high_glucose'] + data['high_cholesterol']
    data['lifestyle_risk_score'] = data['is_overworking'] + data['activity_sleep_mismatch'] + data['oversleep_low_activity']
    data['age_group'] = pd.qcut(data['age'], 5, labels=False, duplicates='drop')
    return data


train = add_features(train)
test = add_features(test)
train = add_reinstated_features(train)
test = add_reinstated_features(test)

activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}
train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

for col in ['gender', 'smoke_status', 'sleep_pattern']:
    le = LabelEncoder().fit(train[col])
    train[col] = le.transform(train[col])
    unseen = [l for l in np.unique(test[col]) if l not in le.classes_]
    if unseen:
        le.classes_ = np.append(le.classes_, unseen)
    test[col] = le.transform(test[col])

print('파생변수 적용 후 shape - train:', train.shape, '/ test:', test.shape)

파생변수 적용 후 shape - train: (2994, 41) / test: (3000, 40)


In [3]:
# 4. medical_history / family_medical_history 인코딩 방식 두 가지 정의
#    (a) 기존: LabelEncoder  (b) 제안: fold-safe 타겟인코딩(조합 기준 평균 stress_score)
tuned_params = dict(n_estimators=5000, learning_rate=0.03, num_leaves=127,
                     min_child_samples=10, random_state=RANDOM_STATE, verbose=-1)

REINSTATED_COLS = ['is_obese', 'high_glucose', 'high_cholesterol',
                   'metabolic_risk_score', 'lifestyle_risk_score', 'age_group']


def encode_disease_label(tr, va):
    tr, va = tr.copy(), va.copy()
    for col in ['medical_history', 'family_medical_history']:
        le = LabelEncoder().fit(tr[col])
        tr[col] = le.transform(tr[col])
        unseen = [l for l in np.unique(va[col]) if l not in le.classes_]
        if unseen:
            le.classes_ = np.append(le.classes_, unseen)
        va[col] = le.transform(va[col])
    return tr, va


def encode_disease_target(tr, va):
    # 반드시 train fold의 라벨만으로 평균을 계산 (val/test는 label 안 씀 -> 리키지 없음)
    tr, va = tr.copy(), va.copy()
    combo_tr = tr['medical_history'] + '_' + tr['family_medical_history']
    global_mean = tr['stress_score'].mean()
    means = tr.groupby(combo_tr)['stress_score'].mean()
    tr['disease_combo_te'] = combo_tr.map(means).fillna(global_mean)
    combo_va = va['medical_history'] + '_' + va['family_medical_history']
    va['disease_combo_te'] = combo_va.map(means).fillna(global_mean)
    tr = tr.drop(columns=['medical_history', 'family_medical_history'])
    va = va.drop(columns=['medical_history', 'family_medical_history'])
    return tr, va


def run_cv(encode_fn, label, drop_reinstated):
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(train))
    fold_maes = []
    for tr_idx, va_idx in kf.split(train):
        tr_df, va_df = train.iloc[tr_idx].copy(), train.iloc[va_idx].copy()
        if drop_reinstated:
            tr_df = tr_df.drop(columns=REINSTATED_COLS)
            va_df = va_df.drop(columns=REINSTATED_COLS)
        tr_df, va_df = encode_fn(tr_df, va_df)

        x_tr = tr_df.drop(columns=['ID', 'stress_score'])
        y_tr = tr_df['stress_score']
        x_va = va_df.drop(columns=['ID', 'stress_score'])
        y_va = va_df['stress_score']

        m = LGBMRegressor(**tuned_params)
        m.fit(x_tr, y_tr, eval_set=[(x_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
        pred = m.predict(x_va)
        oof[va_idx] = pred
        fold_maes.append(mean_absolute_error(y_va, pred))

    cv_mae = mean_absolute_error(train['stress_score'], oof)
    print(f'{label}: fold MAE={[round(v, 4) for v in fold_maes]}, 평균 CV MAE={cv_mae:.4f} (+/- {np.std(fold_maes):.4f})')
    return cv_mae

In [4]:
# 5. 4가지 조합 비교 (A=기존 baseline 기준)
print('=== 4가지 조합 5-Fold CV 비교 ===')
mae_a = run_cv(encode_disease_label, 'A. 기존 19개 (baseline)', drop_reinstated=True)
mae_b = run_cv(encode_disease_label, 'B. 기존19 + 제외피처6 재투입', drop_reinstated=False)
mae_c = run_cv(encode_disease_target, 'C. 기존19 + 타겟인코딩', drop_reinstated=True)
mae_d = run_cv(encode_disease_target, 'D. 기존19 + 제외피처6 + 타겟인코딩(결합)', drop_reinstated=False)

print()
print('요약 (baseline 대비 변화):')
print(f'A baseline         : {mae_a:.4f}')
print(f'B +제외피처6        : {mae_b:.4f} ({mae_b - mae_a:+.4f})')
print(f'C +타겟인코딩        : {mae_c:.4f} ({mae_c - mae_a:+.4f})')
print(f'D +둘 다 결합        : {mae_d:.4f} ({mae_d - mae_a:+.4f})')

=== 4가지 조합 5-Fold CV 비교 ===
A. 기존 19개 (baseline): fold MAE=[0.1757, 0.1662, 0.1767, 0.1768, 0.1747], 평균 CV MAE=0.1740 (+/- 0.0040)
B. 기존19 + 제외피처6 재투입: fold MAE=[0.1764, 0.166, 0.1762, 0.1751, 0.1738], 평균 CV MAE=0.1735 (+/- 0.0039)
C. 기존19 + 타겟인코딩: fold MAE=[0.176, 0.1661, 0.1754, 0.1783, 0.1708], 평균 CV MAE=0.1733 (+/- 0.0044)
D. 기존19 + 제외피처6 + 타겟인코딩(결합): fold MAE=[0.1775, 0.1664, 0.1779, 0.1779, 0.1765], 평균 CV MAE=0.1752 (+/- 0.0044)

요약 (baseline 대비 변화):
A baseline         : 0.1740
B +제외피처6        : 0.1735 (-0.0005)
C +타겟인코딩        : 0.1733 (-0.0007)
D +둘 다 결합        : 0.1752 (+0.0012)


## 결과 해석

- B, C 둘 다 baseline보다 소폭 개선 (-0.0005 ~ -0.0007). 방향은 일관되게 좋아지는 쪽.
- **D(둘 다 결합)는 오히려 baseline보다 나빠짐 (+0.0012).** 제안서에서는 "둘 다 반영하면 더 좋아질 것"으로
  예상했었는데, 실제로 같이 넣으면 상쇄/충돌하는 것으로 보임 — `disease_combo_te`와 `metabolic_risk_score` 등이
  일부 겹치는 정보를 다른 방식으로 인코딩하면서 트리 분기가 오히려 헷갈리는 것으로 추정됩니다 (확정 아님).
- 참고: fold별 표준편차가 0.004 수준이라 B/C/D 사이 차이(0.0005~0.0019)는 로컬 CV만으로는 확실히 가르기
  어려운 크기입니다. **결론은 "B, C 둘 다 시도해볼 가치는 있지만 같이 쓰지는 말자"** 정도로 보는 게 안전합니다.
- 가장 낮은 C안(타겟인코딩)을 최종 후보로 채택해 아래에서 제출 파일을 만듭니다.

In [5]:
# 6. 최종 모델: C안(타겟인코딩)을 fold-safe 방식으로 재확인하며 test 예측까지 생성
reinstated_cols = REINSTATED_COLS
train_final = train.drop(columns=reinstated_cols).copy()
test_final = test.drop(columns=reinstated_cols).copy()

kf_final = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
test_pred = np.zeros(len(test_final))
oof_final = np.zeros(len(train_final))

for tr_idx, va_idx in kf_final.split(train_final):
    tr_df = train_final.iloc[tr_idx].copy()
    va_df = train_final.iloc[va_idx].copy()
    te_df = test_final.copy()

    combo_tr = tr_df['medical_history'] + '_' + tr_df['family_medical_history']
    global_mean = tr_df['stress_score'].mean()
    means = tr_df.groupby(combo_tr)['stress_score'].mean()
    tr_df['disease_combo_te'] = combo_tr.map(means).fillna(global_mean)
    for d in (va_df, te_df):
        combo = d['medical_history'] + '_' + d['family_medical_history']
        d['disease_combo_te'] = combo.map(means).fillna(global_mean)
    for d in (tr_df, va_df, te_df):
        d.drop(columns=['medical_history', 'family_medical_history'], inplace=True)

    x_tr = tr_df.drop(columns=['ID', 'stress_score'])
    y_tr = tr_df['stress_score']
    x_va = va_df.drop(columns=['ID', 'stress_score'])
    y_va = va_df['stress_score']
    x_te = te_df.drop(columns=['ID'])

    m = LGBMRegressor(**tuned_params)
    m.fit(x_tr, y_tr, eval_set=[(x_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_final[va_idx] = m.predict(x_va)
    test_pred += np.clip(m.predict(x_te), 0, 1) / kf_final.n_splits

final_cv_mae = mean_absolute_error(train_final['stress_score'], oof_final)
print(f'최종 CV MAE (#16, 타겟인코딩): {final_cv_mae:.4f}')

os.makedirs('../submissions', exist_ok=True)
sample_submission['stress_score'] = test_pred
submit_path = '../submissions/submit_16_target_encoding.csv'
sample_submission.to_csv(submit_path, index=False)
print(f'제출 파일 저장: {submit_path}')

최종 CV MAE (#16, 타겟인코딩): 0.1733
제출 파일 저장: ../submissions/submit_16_target_encoding.csv
